# Clasificación UNSPSC de todos los indicadores priorizados (2 fases)

El objetivo del presente código consiste en estructurar la relación semántica entre 
la descripción de los indicadores objeto de costeo (su descripción textual) y la estructura
UNSPC bajo la cual se clasifican los contratos del SECOP-II en categorías de Segmento, Familia y 
Producto. Se procede de esta manera ya que cada segmento tendrá un factor de ajuste territorial 
calculado, en distintas categorías de tipología de producto, permitiendo con ello la subregionalización
directa del costeo de cada contrato asociado a cada indicador. 

Este notebook readapta `clasificar_indicadores_2fases.py` para correr en Jupyter
con control manual sobre qué partes se ejecutan y cuáles solo se cargan/reportan.

**A diferencia de la versión "equipo"**, - desarrollada para la estimación espejo-,
este parte de **todos los indicadores
posibles** en `data_priorizados_Indicadores.xlsx` (hoja `Indicador_asociado`,
605 filas) — no de una muestra de sólo tres indicadores.

**Dos correcciones sobre el script original** (documentadas para que sepas qué
cambió y por qué):

1. **Deduplicación antes de clasificar.** El texto del indicador se usa como
   llave para unir Fase 1 con Fase 2 (`merge(..., on="indicador")`). De las 605
   filas, 7 tienen el mismo texto repetido — sin deduplicar, ese `merge`
   multiplica filas para esos 7 casos. Aquí se clasifica solo el texto único
   (598 valores), y luego se reparte el resultado a todas las filas originales
   que comparten ese texto. Efecto extra: menos llamadas al LLM.
2. **Se conserva `Cod_indicador`** en la salida final. El script original sólo
   usaba el texto (columna B) y descartaba todo lo demás. Nota: 126 de las 605 filas tienen
   `Cod_indicador = "Sin código"` (indicadores MGA u otros sin código PDET
   formal) — quedan igual en la salida y no se descartan.

**Control de ejecución:** las dos fases (LLM) tienen su propia bandera
(`EJECUTAR_FASE1`, `EJECUTAR_FASE2`) y su propio checkpoint en disco. Puede
correr Fase 1 hoy, dejar `False` mañana y solo re-correr Fase 2 si cambia la
jerarquía, sin gastar tokens de nuevo en Fase 1.

## Revisión de las bases de datos

La base de datos `data_priorizados_Indicadores.xlsx` contiene los indicadores completos, con su respectiva descripción,
y no solo una muestra de indicadores priorizados.

## Base Indicadores Priorizados Completos 

In [3]:
import pandas as pd

# Cargando la base de datos de Priorizados completa (recuerda poner en tu ruta estos archivos ) 
ruta = r'D:\Consultorias_2026\Estructura_General\ART-Costos_Observados\Resultados_Finales\data_priorizados_Indicadores.xlsx'
base_indicadores = pd.read_excel(ruta)
base_indicadores

,Subregión,Capítulo,CodSubprograma,Subprograama,subpro_econome,numresult,prioridadguillermo,Cod_indicador,Indicador_asociado,Tipo_indicador,Indicador Lider Programa,Indicador lider Subprograma,Fuente General
0,NaN,NaN,NaN,NaN,15,26,7,NaN,NaN,NaN,NaN,NaN,NaN
1,Alto PatÃ­a,Intercultural,G01010101,NaN,0,0,1,LB_3.2.6,Tasa de cobertura bruta de educación preescolar,Resultado,No,No,Linea Base
2,Alto PatÃ­a,Intercultural,G01010101,NaN,0,0,1,LB_10.2.3,Indicadores simples del IPM - Inasistencia esc...,Resultado,No,No,Linea Base
3,Alto PatÃ­a,Intercultural,G01010101,NaN,0,0,1,LB_10.2.4,Indicadores simples del IPM - Rezago escolar,Resultado,No,No,Linea Base
4,Alto PatÃ­a,Intercultural,G01010101,NaN,0,0,1,LB_3.2.1,Tasa de asistencia escolar de personas entre 6...,Resultado,No,No,Linea Base
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9908,UrabÃ¡ AntioqueÃ±o,Ãtnico,NE16060203,NaN,0,0,2,P6.74.,Tasa de personas que acceden a programas e ini...,Resultado,Si,Si,Bateria PATR
9909,UrabÃ¡ AntioqueÃ±o,Ãtnico,NE16060203,NaN,0,0,2,450201801,Procesos colectivos de memoria histórica y arc...,Producto,No,No,MGA
9910,UrabÃ¡ AntioqueÃ±o,Ãtnico,NE16060203,NaN,0,0,2,410103805,Procesos de Memoria Histórica con poblaciones ...,Producto,No,No,MGA
9911,UrabÃ¡ AntioqueÃ±o,Ãtnico,NE16060203,NaN,0,0,2,021200901,Servicio de investigación de reconstrucción de...,Producto,No,No,MGA


 Esta base contiene mas de 500 indicadores unicos, clasificados entre indicadores de producto y resultado, asi como su correspondencia 
 con las lineas y subprogramas que integran la estructura programática.

In [15]:
# Considerando por descripción del indicador y no su codigo 

indicadores = pd.crosstab(index=base_indicadores['Indicador_asociado'], columns='Frecuencia')
print("El número de indicadores únicos con descripción es: " + str(indicadores.shape[0]))

El numero de indicadores únicos es: 598


## Base de datos de Jerarquía Complta

Por su parte, la base de datos correspondiente a 'jerarquia_completa.xlsx' contiene las clasificaciones propias de SECOP-II para 
los tipos Segmento, Familia y Producto. 


In [16]:
ruta_2 = r'D:\Consultorias_2026\Estructura_General\ART-Costos_Observados\Resultados_Finales\jerarquia_completa.xlsx'
base_tipologia = pd.read_excel(ruta_2)
base_tipologia

,nombre_segmento,nombre_familia,nombre_clase,nombre_producto
0,"Alimentos, Bebidas y Tabaco",Aceites y grasas comestibles,Grasas y aceites animales comestibles,Aceites animal comestibles
1,"Alimentos, Bebidas y Tabaco",Aceites y grasas comestibles,Grasas y aceites animales comestibles,Grasa saturada animal comestibles
2,"Alimentos, Bebidas y Tabaco",Aceites y grasas comestibles,Grasas y aceites animales comestibles,Grasas y aceites animales comestibles
3,"Alimentos, Bebidas y Tabaco",Aceites y grasas comestibles,Grasas y aceites vegetales comestibles,Aceites vegetales o de planta comestibles
4,"Alimentos, Bebidas y Tabaco",Aceites y grasas comestibles,Grasas y aceites vegetales comestibles,Grasas saturadas de vegetales o plantas comest...
...,...,...,...,...
12727,"Vehículos Comerciales, Militares y Particulare...",Vehículos de motor,Vehículos especializados o de recreo,Van para radiodifusión externa
12728,"Vehículos Comerciales, Militares y Particulare...",Vehículos de motor,Vehículos especializados o de recreo,Vehículo eléctrico de vecindario
12729,"Vehículos Comerciales, Militares y Particulare...",Vehículos de motor,Vehículos especializados o de recreo,Vehículos anfibios
12730,"Vehículos Comerciales, Militares y Particulare...",Vehículos de motor,Vehículos especializados o de recreo,Vehículos especializados o de recreo


Esta base clasifica por descripción de los segmentos, familias y productos y viene de la base de datos SECOP-II dipuesta para el análisis (no a partir de la API socrata). Se opta, como es natural, por la descripción de estas categorías y no su codigo, para ejecutar el proceso de clasificación. semántica. Como puede notarse, la mayor desagregación posible está a nivel de producto. 

In [18]:
segmentos = pd.crosstab(index=base_tipologia['nombre_segmento'], columns='Frecuencia')
familias = pd.crosstab(index=base_tipologia['nombre_familia'], columns='Frecuencia')
productos = pd.crosstab(index=base_tipologia['nombre_producto'], columns = 'Frecuencia')
print("El número de segmentos únicos con descripción es: " + str(segmentos.shape[0]))
print("El número de familias únicos con descripción es: " + str(familias.shape[0]))
print("El número de productos únicos con descripción es: " + str(productos.shape[0]))

El número de segmentos únicos con descripción es: 57
El número de familias únicos con descripción es: 405
El número de productos únicos con descripción es: 12697


## Configuración

Cambia aquí lo que necesites. `EJECUTAR_FASE1` / `EJECUTAR_FASE2` en `False`
significa "no llamar al LLM, cargar el checkpoint ya guardado". Ponlas en
`True` solo cuando quieras (re)generar esa fase específica.

In [1]:
import os, json, time
from pathlib import Path
from collections import defaultdict

import pandas as pd

# ══════════════════════════════════════════════════════════
#  CONTROL DE EJECUCIÓN — lo único que normalmente hay que tocar
# ══════════════════════════════════════════════════════════
EJECUTAR_FASE1 = False   # True = llama al LLM para Indicador -> Segmento
EJECUTAR_FASE2 = False   # True = llama al LLM para Indicador -> Producto

# El nombre del modelo del script original ("claude-sonnet-4-20250514") es un
# snapshot fechado; aquí uso el alias vigente, pero confírmalo en docs.claude.com
# antes de una corrida grande.
MODELO = "claude-sonnet-4-6"

ARCHIVO_INDICADORES = "data_priorizados_Indicadores.xlsx"
HOJA_INDICADORES     = "Indicador_asociado"
ARCHIVO_JERARQUIA    = "jerarquia_completa.xlsx"

CHECKPOINT_FASE1 = "_checkpoint_fase1_segmento.xlsx"
CHECKPOINT_FASE2 = "_checkpoint_fase2_producto.xlsx"
ARCHIVO_SALIDA    = "indicadores_clasificados_2fases.xlsx"

BATCH_SIZE_F1  = 20
BATCH_SIZE_F2  = 15
PAUSA          = 0.4
MAX_REINTENTOS = 3

## `llamar_api` — la única función que habla con el LLM

Envía el prompt, limpia la respuesta (por si el modelo agrega ```` ```json ````
alrededor), y la parsea como JSON. Si falla (JSON inválido, rate limit, error
de red) reintenta hasta `MAX_REINTENTOS` veces con espera creciente. Si
después de todos los intentos sigue fallando, devuelve `None` — quien la llama
(Fase 1 o Fase 2) decide qué hacer con ese fallo (normalmente: marcar
"Sin clasificar" y seguir, para que un solo lote fallido no tumbe toda la corrida).

In [2]:
def llamar_api(client, prompt, max_tokens=1500):
    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            msg = client.messages.create(
                model=MODELO, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = msg.content[0].text.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"  aviso: JSON invalido (intento {intento}): {e}")
            time.sleep(2 * intento)
        except Exception as e:
            espera = 30 if "rate" in str(e).lower() else 3 * intento
            print(f"  aviso: error (intento {intento}): {e}")
            time.sleep(espera)
    return None

## `cargar_datos` — indicadores únicos + jerarquía UNSPSC

Lee la columna `Indicador_asociado` (605 filas), limpia saltos de línea y
espacios, y **deduplica por texto** (598 valores únicos) — este es el cambio
respecto al script original, explicado en la portada. También guarda
`df_original` completo (con `Cod_indicador`) para poder repartir la
clasificación a todas las filas al final, aunque compartan texto.

La jerarquía (`jerarquia_completa.xlsx`) no cambia: 57 segmentos, ~12 700
productos.

In [3]:
def cargar_datos():
    df_original = pd.read_excel(ARCHIVO_INDICADORES, sheet_name=HOJA_INDICADORES)
    col_b = df_original.columns[1]  # 'Indicador_asociado'
    df_original["_texto_limpio"] = (
        df_original[col_b].astype(str).str.replace(r"\n", " ", regex=True).str.strip()
    )
    df_original = df_original[
        df_original["_texto_limpio"].notna()
        & (df_original["_texto_limpio"].str.lower() != "nan")
        & (df_original["_texto_limpio"] != "")
    ]

    indicadores_unicos = sorted(df_original["_texto_limpio"].drop_duplicates().tolist())

    df_jer = (
        pd.read_excel(ARCHIVO_JERARQUIA)
        [["nombre_segmento", "nombre_familia", "nombre_clase", "nombre_producto"]]
        .dropna().drop_duplicates()
    )

    print(f"Filas originales (data_priorizados_Indicadores): {len(df_original)}")
    print(f"Textos unicos a clasificar:                      {len(indicadores_unicos)}")
    print(f"Segmentos unicos:                                {df_jer['nombre_segmento'].nunique()}")
    print(f"Productos unicos:                                {df_jer['nombre_producto'].nunique()}")
    return df_original, indicadores_unicos, df_jer


df_original, indicadores_unicos, df_jer = cargar_datos()

Filas originales (data_priorizados_Indicadores): 605
Textos unicos a clasificar:                      598
Segmentos unicos:                                57
Productos unicos:                                12697


## Fase 1 — Indicador → Segmento

Clasifica cada indicador (texto) al segmento UNSPSC más cercano, en lotes de
`BATCH_SIZE_F1` (20). El catálogo de 57 segmentos es fijo y pequeño, así que
cabe entero en cada prompt sin problema de tokens.

Si un lote falla tras los reintentos, esos indicadores quedan marcados
`"Sin clasificar"` — no detiene la corrida completa.

In [4]:
def prompt_fase1(batch, segmentos_txt):
    batch_txt = "\n".join(f"{i+1}. {ind}" for i, ind in enumerate(batch))
    return f"""Eres experto en clasificacion de bienes y servicios del sector publico colombiano \
usando el estandar UNSPSC adaptado por Colombia Compra Eficiente para SECOP II.

SEGMENTOS DISPONIBLES:
{segmentos_txt}

Asocia cada indicador de politica publica PDET con el SEGMENTO semanticamente mas cercano.
Reglas:
1. Usa EXACTAMENTE el texto del segmento tal como aparece en la lista.
2. Si ninguno es razonablemente cercano, escribe "Sin clasificar".
3. Responde UNICAMENTE con un JSON array valido, sin texto adicional ni backticks.

INDICADORES:
{batch_txt}

Formato (mismo orden que los indicadores):
[{{"indicador":"texto exacto","segmento":"segmento exacto"}}]"""


def fase1(client, indicadores, df_jer):
    segmentos_txt = "\n".join(sorted(df_jer["nombre_segmento"].unique().tolist()))
    batches = [indicadores[i:i + BATCH_SIZE_F1] for i in range(0, len(indicadores), BATCH_SIZE_F1)]
    results = []
    print(f"FASE 1: {len(batches)} lote(s) de hasta {BATCH_SIZE_F1}")
    for i, batch in enumerate(batches):
        print(f"  lote {i+1:02d}/{len(batches)} ({len(batch)} ind.)...", end=" ")
        data = llamar_api(client, prompt_fase1(batch, segmentos_txt), max_tokens=max(1600, len(batch) * 80))
        if data:
            results.extend(data)
            print(f"ok ({len(data)})")
        else:
            results.extend({"indicador": ind, "segmento": "Sin clasificar"} for ind in batch)
            print("fallido -> Sin clasificar")
        time.sleep(PAUSA)
    return pd.DataFrame(results)[["indicador", "segmento"]]

### Ejecutar o cargar Fase 1

Con `EJECUTAR_FASE1 = False` (default) carga el checkpoint ya guardado. Con
`True`, llama al LLM (requiere `ANTHROPIC_API_KEY` en el entorno) y sobrescribe
el checkpoint.

In [10]:
| eval: true
| output: true

if EJECUTAR_FASE1:
    import anthropic
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("Falta ANTHROPIC_API_KEY en el entorno.")
    client = anthropic.Anthropic(api_key=api_key)
    df_f1 = fase1(client, indicadores_unicos, df_jer)
    df_f1.to_excel(CHECKPOINT_FASE1, index=False)
    print(f"Checkpoint guardado: {CHECKPOINT_FASE1}")
else:
    if not Path(CHECKPOINT_FASE1).exists():
        raise FileNotFoundError(
            f"No existe '{CHECKPOINT_FASE1}' y EJECUTAR_FASE1=False. "
            f"Pon EJECUTAR_FASE1=True (con tu API key) para generarlo la primera vez."
        )
    df_f1 = pd.read_excel(CHECKPOINT_FASE1)
    print(f"Cargado desde checkpoint: {CHECKPOINT_FASE1} ({len(df_f1)} filas)")

sc_f1 = (df_f1["segmento"] == "Sin clasificar").sum()
print(f"Clasificados en Fase 1: {len(df_f1) - sc_f1}/{len(df_f1)}")

SyntaxError: invalid syntax (990715120.py, line 1)

## Fase 2 — Indicador → Producto (dentro del segmento ya asignado)

Agrupa los indicadores por el segmento que les tocó en Fase 1, y por cada
segmento arma un catálogo **solo con los productos de ese segmento** (en vez
de los ~12 700 productos totales). Esto es lo que hace viable el costo en
tokens: el catálogo por segmento es, en promedio, ~28 veces más pequeño que el
catálogo completo.

In [6]:
def prompt_fase2(batch, productos_txt, segmento):
    batch_txt = "\n".join(f"{i+1}. {ind}" for i, ind in enumerate(batch))
    return f"""Eres experto en clasificacion de bienes y servicios del sector publico colombiano \
usando el estandar UNSPSC adaptado por Colombia Compra Eficiente para SECOP II.

SEGMENTO: {segmento}

PRODUCTOS DISPONIBLES EN ESTE SEGMENTO:
{productos_txt}

Asocia cada indicador con el PRODUCTO mas especifico y semanticamente mas cercano.
Reglas:
1. Usa EXACTAMENTE el texto del producto tal como aparece en la lista.
2. Si ninguno es razonablemente cercano, escribe "Sin clasificar".
3. Responde UNICAMENTE con un JSON array valido, sin texto adicional ni backticks.

INDICADORES:
{batch_txt}

Formato (mismo orden que los indicadores):
[{{"indicador":"texto exacto","producto":"producto exacto"}}]"""


def fase2(client, df_f1, df_jer):
    prod_por_seg = (
        df_jer.groupby("nombre_segmento")["nombre_producto"]
        .apply(lambda s: sorted(s.dropna().unique().tolist())).to_dict()
    )
    grupos = defaultdict(list)
    for _, row in df_f1.iterrows():
        grupos[row["segmento"]].append(row["indicador"])

    results = []
    print(f"FASE 2: {len(grupos)} segmento(s) distintos")
    for idx, (segmento, indicadores_seg) in enumerate(sorted(grupos.items()), start=1):
        if segmento == "Sin clasificar":
            results.extend({"indicador": ind, "producto": "Sin clasificar"} for ind in indicadores_seg)
            print(f"  [{idx:02d}] Sin clasificar ({len(indicadores_seg)} ind.) -> omitido")
            continue
        productos = prod_por_seg.get(segmento, [])
        if not productos:
            results.extend({"indicador": ind, "producto": "Sin clasificar"} for ind in indicadores_seg)
            continue
        productos_txt = "\n".join(productos)
        batches = [indicadores_seg[i:i + BATCH_SIZE_F2] for i in range(0, len(indicadores_seg), BATCH_SIZE_F2)]
        print(f"  [{idx:02d}] {segmento[:48]} - {len(indicadores_seg)} ind. | {len(productos)} prod. | {len(batches)} lote(s)")
        for j, batch in enumerate(batches):
            print(f"    lote {j+1}/{len(batches)} ({len(batch)} ind.)...", end=" ")
            data = llamar_api(client, prompt_fase2(batch, productos_txt, segmento), max_tokens=max(600, len(batch) * 60))
            if data:
                results.extend(data)
                print(f"ok ({len(data)})")
            else:
                results.extend({"indicador": ind, "producto": "Sin clasificar"} for ind in batch)
                print("fallido -> Sin clasificar")
            time.sleep(PAUSA)
    return pd.DataFrame(results)[["indicador", "producto"]]

### Ejecutar o cargar Fase 2

Igual que Fase 1: `False` carga el checkpoint, `True` vuelve a llamar al LLM.
Puedes tener `EJECUTAR_FASE1=False` y `EJECUTAR_FASE2=True` al mismo tiempo
— por ejemplo si quieres reclasificar solo productos porque actualizaste
`jerarquia_completa.xlsx`, sin volver a gastar tokens en segmentos.

In [7]:
if EJECUTAR_FASE2:
    import anthropic
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("Falta ANTHROPIC_API_KEY en el entorno.")
    client = anthropic.Anthropic(api_key=api_key)
    df_f2 = fase2(client, df_f1, df_jer)
    df_f2.to_excel(CHECKPOINT_FASE2, index=False)
    print(f"Checkpoint guardado: {CHECKPOINT_FASE2}")
else:
    if not Path(CHECKPOINT_FASE2).exists():
        raise FileNotFoundError(
            f"No existe '{CHECKPOINT_FASE2}' y EJECUTAR_FASE2=False. "
            f"Pon EJECUTAR_FASE2=True (con tu API key) para generarlo la primera vez."
        )
    df_f2 = pd.read_excel(CHECKPOINT_FASE2)
    print(f"Cargado desde checkpoint: {CHECKPOINT_FASE2} ({len(df_f2)} filas)")

sc_f2 = (df_f2["producto"] == "Sin clasificar").sum()
print(f"Clasificados en Fase 2: {len(df_f2) - sc_f2}/{len(df_f2)}")

FileNotFoundError: No existe '_checkpoint_fase2_producto.xlsx' y EJECUTAR_FASE2=False. Pon EJECUTAR_FASE2=True (con tu API key) para generarlo la primera vez.

## `enriquecer` y `guardar` — unir todo y exportar

`enriquecer` une Fase 1 + Fase 2 por texto de indicador (ahora seguro, porque
ambas se calcularon sobre la lista ya deduplicada), trae `Familia`/`Clase`
desde la jerarquía, y **reparte el resultado a las 605 filas originales**
(incluyendo `Cod_indicador`) haciendo el join en la dirección
`df_original -> clasificación` — así ningún texto duplicado explota filas.

`guardar` exporta a Excel con formato (encabezado azul, columnas anchas) igual
que el script original.

In [8]:
def enriquecer(df_original, df_f1, df_f2, df_jer):
    clasif = df_f1.merge(df_f2, on="indicador", how="left")

    lkp = (
        df_jer.drop_duplicates(subset="nombre_producto")
        [["nombre_producto", "nombre_familia", "nombre_clase"]]
    )
    clasif = clasif.merge(lkp, left_on="producto", right_on="nombre_producto", how="left")
    clasif = clasif.drop(columns=["nombre_producto"])
    for col in ["nombre_familia", "nombre_clase"]:
        clasif[col] = clasif[col].fillna("Sin clasificar")

    # Reparto seguro: LEFT join desde el original (605 filas) hacia la
    # clasificacion unica (598 filas) -- nunca multiplica filas.
    df = df_original.merge(clasif, left_on="_texto_limpio", right_on="indicador", how="left")

    df = df.rename(columns={
        "Cod_indicador": "Codigo_indicador",
        "_texto_limpio": "Indicador",
        "segmento": "Segmento",
        "producto": "Producto",
        "nombre_familia": "Familia",
        "nombre_clase": "Clase",
    })
    return df[["Codigo_indicador", "Indicador", "Segmento", "Familia", "Clase", "Producto"]]


def guardar(df_final, path):
    from openpyxl.styles import Font, PatternFill, Alignment
    col_widths = {"A": 16, "B": 70, "C": 45, "D": 45, "E": 45, "F": 55}
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df_final.to_excel(writer, index=False, sheet_name="Clasificacion")
        ws = writer.sheets["Clasificacion"]
        fill = PatternFill("solid", start_color="1E3A5F", end_color="1E3A5F")
        font = Font(bold=True, name="Arial", size=11, color="FFFFFF")
        align = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for letra, ancho in col_widths.items():
            ws.column_dimensions[letra].width = ancho
            cell = ws[f"{letra}1"]
            cell.font, cell.fill, cell.alignment = font, fill, align
        ws.row_dimensions[1].height = 22
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
    sc = (df_final["Producto"] == "Sin clasificar").sum()
    print(f"Guardado: {path}")
    print(f"  Total: {len(df_final)} | Clasificados: {len(df_final)-sc} | Sin clasificar: {sc}")

### Ensamblar y guardar (solo si se ejecutó alguna fase de nuevo)

Si ninguna de las dos fases se re-ejecutó en esta sesión, se carga
directamente `ARCHIVO_SALIDA` si ya existe (no hace falta recalcular nada
solo para ver el reporte).

In [ ]:
if EJECUTAR_FASE1 or EJECUTAR_FASE2:
    df_final = enriquecer(df_original, df_f1, df_f2, df_jer)
    guardar(df_final, ARCHIVO_SALIDA)
elif Path(ARCHIVO_SALIDA).exists():
    df_final = pd.read_excel(ARCHIVO_SALIDA, sheet_name="Clasificacion")
    print(f"Cargado desde: {ARCHIVO_SALIDA} ({len(df_final)} filas)")
else:
    df_final = enriquecer(df_original, df_f1, df_f2, df_jer)
    guardar(df_final, ARCHIVO_SALIDA)

## Reporte de resultados

In [ ]:
import matplotlib.pyplot as plt

sin_clasificar = (df_final["Producto"] == "Sin clasificar").sum()
clasificados = len(df_final) - sin_clasificar
sin_codigo = (df_final["Codigo_indicador"] == "Sin código").sum()

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["Clasificados", "Sin clasificar"], [clasificados, sin_clasificar], color=["#1E3A5F", "#C0392B"])
ax.set_ylabel("N de indicadores")
for i, v in enumerate([clasificados, sin_clasificar]):
    ax.text(i, v + 1, str(v), ha="center", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Total de filas: {len(df_final)}")
print(f"Clasificados: {clasificados} ({clasificados/len(df_final):.0%})")
print(f"Sin clasificar: {sin_clasificar}")
print(f"Con Codigo_indicador = 'Sin codigo' (indicadores sin codigo PDET formal): {sin_codigo}")

In [ ]:
conteo_seg = df_final["Segmento"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, max(3, 0.28 * len(conteo_seg))))
ax.barh(conteo_seg.index.astype(str), conteo_seg.values, color="#2E86AB")
ax.set_xlabel("N de indicadores")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
df_final.sort_values("Codigo_indicador", key=lambda s: s.astype(str))